**Avvio locale e Colab:** preparare la cartella completa seguendo il [README del laboratorio](README.md). Conservare gli identificativi delle celle e salvare gli output nel proprio runtime.

**C01**

# Laboratorio 7 — Due o più viste

**Computer Vision, LM Ingegneria Informatica**

Obiettivo: trasformare le relazioni della dispensa in esperimenti
riproducibili. Il laboratorio usa una scena sintetica con verità nota, così
possiamo distinguere errore di misura, outlier, degenerazione e convenzioni.

Al termine saprai:

1. verificare una matrice fondamentale con residui e copertura;
2. osservare l'effetto della soglia RANSAC e degli outlier;
3. recuperare posa, cheirality e punti 3D;
4. collegare disparità, baseline, distanza e incertezza;
5. riconoscere l'ambiguità del matching stereo;
6. leggere un piccolo bundle adjustment come rete camera–punto.

**Regola di laboratorio:** prima prevedi il risultato, poi esegui, infine
spiega citando unità, soglie e statistiche. Il seed globale è fissato a 7.

**Uso:** esercitazione autonoma sui metodi geometrici e sulle loro condizioni di validità.

Nei materiali del corso si usa il **punto decimale** (0.5) e la **virgola seguita da uno spazio** fra componenti di vettori e argomenti, per esempio (1.2, 1, −0.3). I conteggi interi non usano separatori delle migliaia.

**C02**

## 0. Preparazione

Il notebook richiede NumPy, SciPy, Matplotlib e OpenCV. Le corrispondenze dei blocchi epipolari sono pixel senza distorsione. Usiamo la stessa notazione della dispensa e delle figure:

- $P$ (o $P_j$) è il punto 3D; $P^{(L)}$ e $P^{(R)}$ sono le coordinate dello stesso punto nei frame sinistro e destro. $C_L, C_R$ sono i centri delle camere.
- $p_L=(u_L, v_L)^\mathsf T$ e $p_R=(u_R, v_R)^\mathsf T$ sono i pixel 2D; $\widetilde p_L=(u_L, v_L, 1)^\mathsf T$ e $\widetilde p_R=(u_R, v_R, 1)^\mathsf T$ sono i pixel omogenei.
- $x_L=K_L^{-1}\widetilde p_L$ e $x_R=K_R^{-1}\widetilde p_R$ sono i raggi calibrati. $\Pi_L, \Pi_R$ sono le matrici di proiezione; $e_L, e_R$ gli epipoli e $\ell_L, \ell_R$ le rette epipolari.

La posa segue $P^{(R)}=RP^{(L)}+t$. Nel riferimento sinistro $C_L=0$ e $C_R=-R^\mathsf Tt$: la baseline fra centri non è direttamente t.

**F è la matrice fondamentale**, che impone $\widetilde p_R^\mathsf TF\widetilde p_L=0$ ai pixel omogenei. **E è la matrice essenziale**, che impone $x_R^\mathsf TEx_L=0$ ai raggi calibrati. Con $E=[t]_\times R$, vale $F=K_R^{-\mathsf T}EK_L^{-1}$. La retta candidata nella vista destra è $\ell_R=F\widetilde p_L$: non è ancora il pixel omologo.

**Ponte con i nomi del codice.** `x1`/`x2` sono array dei pixel p_L/p_R, non raggi normalizzati; `x1h`/`x2h` sono i pixel omogenei. `X_true`, `X_tri`, `X_aff` e `X_ba_true` contengono punti 3D P_j, mentre `P1`/`P2` sono matrici di proiezione Π_L/Π_R. La matrice `K` è condivisa dalle due camere sintetiche e viene passata esplicitamente alla stima essenziale. X/Y/Z sugli assi sono componenti cartesiane. Nei blocchi multivista p_ij è l'osservazione del punto j nella vista i, con n_v viste e n_p punti; `x_aff_i` contiene coordinate 2D in unità sintetiche. La camera 3 del mini BA conserva il suo indice.

**Simboli delle ultime derivazioni.** In C19 la triangolazione corrisponde a righe q_ikᵀ della matrice Π_i, ciascuna con quattro componenti. Nel blocco affine le coordinate centrate sono p_ij^c=p_ij−p̄_i; W_c ha dimensione 2n_v×n_p. Questi simboli descrivono gli stessi oggetti del codice già presente.

In [ ]:
# C03
import sys
import numpy as np
import cv2
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from scipy.spatial.transform import Rotation

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(7)
cv2.setRNGSeed(7)

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('OpenCV:', cv2.__version__)
print('Seed globale: 7')

In [ ]:
# C04
def skew(t):
    tx, ty, tz = np.asarray(t).reshape(3)
    return np.array([[0, -tz, ty], [tz, 0, -tx], [-ty, tx, 0]], float)

def project(K, R, t, X):
    """Proietta X (N,3) con X_cam = R X + t."""
    Xc = (R @ X.T + np.asarray(t).reshape(3, 1)).T
    uvw = (K @ Xc.T).T
    return uvw[:, :2] / uvw[:, 2:3], Xc[:, 2]

def to_h(x):
    return np.c_[x, np.ones(len(x))]

def sampson(F, x1, x2):
    x1h, x2h = to_h(x1), to_h(x2)
    Fx1 = (F @ x1h.T).T
    Ftx2 = (F.T @ x2h.T).T
    num = np.sum(x2h * Fx1, axis=1) ** 2
    den = Fx1[:, 0]**2 + Fx1[:, 1]**2 + Ftx2[:, 0]**2 + Ftx2[:, 1]**2
    return num / np.maximum(den, 1e-12)

def rotation_error_deg(R_est, R_true):
    R_delta = R_est @ R_true.T
    c = np.clip((np.trace(R_delta) - 1) / 2, -1, 1)
    return np.degrees(np.arccos(c))

def direction_error_deg(a, b):
    a, b = np.ravel(a), np.ravel(b)
    c = np.clip(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)), -1, 1)
    return np.degrees(np.arccos(c))

def line_segment(line, width, height):
    a, b, c = line
    pts = []
    if abs(b) > 1e-9:
        for x in (0, width - 1):
            y = -(a*x + c) / b
            if 0 <= y < height:
                pts.append((x, y))
    if abs(a) > 1e-9:
        for y in (0, height - 1):
            x = -(b*y + c) / a
            if 0 <= x < width:
                pts.append((x, y))
    return pts[:2]

**C05**

## 1. Scena sintetica, rumore e outlier

Creiamo due camere calibrate, punti davanti a entrambe e corrispondenze
rumorose. Una frazione delle osservazioni nella seconda immagine viene
sostituita con pixel casuali: conosciamo quindi la vera maschera degli inlier.

In [ ]:
# C06
W, H = 960, 640
K = np.array([[800., 0., W/2], [0., 800., H/2], [0., 0., 1.]])
R_true = Rotation.from_euler('y', 4.0, degrees=True).as_matrix()
t_true = np.array([-0.22, 0.015, 0.01])

N = 180
X_true = np.column_stack([
    rng.uniform(-1.7, 1.7, N),
    rng.uniform(-1.0, 1.0, N),
    rng.uniform(3.0, 8.0, N),
])
x1_clean, z1 = project(K, np.eye(3), np.zeros(3), X_true)
x2_clean, z2 = project(K, R_true, t_true, X_true)
visible = ((x1_clean[:, 0] > 15) & (x1_clean[:, 0] < W-15) &
           (x1_clean[:, 1] > 15) & (x1_clean[:, 1] < H-15) &
           (x2_clean[:, 0] > 15) & (x2_clean[:, 0] < W-15) &
           (x2_clean[:, 1] > 15) & (x2_clean[:, 1] < H-15) &
           (z1 > 0) & (z2 > 0))
X_true, x1_clean, x2_clean = X_true[visible], x1_clean[visible], x2_clean[visible]

sigma_px = 0.65
x1 = x1_clean + rng.normal(0, sigma_px, x1_clean.shape)
x2 = x2_clean + rng.normal(0, sigma_px, x2_clean.shape)
true_inlier = np.ones(len(x1), dtype=bool)
out_idx = rng.choice(len(x1), size=round(0.25*len(x1)), replace=False)
x2[out_idx] = np.column_stack([rng.uniform(0, W, len(out_idx)),
                               rng.uniform(0, H, len(out_idx))])
true_inlier[out_idx] = False

print(f'Punti visibili: {len(x1)}')
print(f'Rumore: sigma={sigma_px:.2f} px')
print(f'Outlier inseriti: {(~true_inlier).sum()} ({(~true_inlier).mean():.1%})')

**C06a**

### Demo guidata — profondità e disparità

La scena di questa demo è quella dell'esempio canonico della dispensa: camere parallele,
$C_L=0$, $C_R=(b, 0, 0)^\mathsf T$, $R=I$, $t=(-b, 0, 0)^\mathsf T$ e punto principale nell'origine.
La focale è **800 px**, la baseline **0.12 m** e il punto è $P=(0.18, 0.10, Z)^\mathsf T$ m.
Si varia soltanto **Z**, mantenendo X e Y fissi; le coordinate delle due immagini sono
misurate rispetto al punto principale, con **v positivo verso il basso**.

$$u_L=\frac{fX}{Z}, \quad u_R=\frac{f(X-b)}{Z}, \quad
v_L=v_R=\frac{fY}{Z}, \quad d=u_L-u_R=\frac{fb}{Z}.$$

Eseguire **C06b** e muovere il controllo **Z (m)** fra 0.75 e 8 m.
La figura aggiorna il punto 3D, le due proiezioni e il marker sulla curva.
Le ordinate restano uguali fra le viste, ma cambiano con la distanza: a Z=1, 2 e 4 m
valgono rispettivamente 80, 40 e 20 px. Il valore iniziale Z=2 m dà
$p_L=(72, 40)$ px, $p_R=(24, 40)$ px e d=48 px.

**Avvio:** C06b importa da sola NumPy, Matplotlib e IPython; non richiede dati,
GPU o esecuzioni precedenti. Gli altri esperimenti continuano a usare la scena rumorosa di C06,
con i propri prerequisiti e il seed originale.
In Google Colab usare il runtime CPU; il codice abilita il gestore dei widget se Colab è rilevato.
Se necessario installare le dipendenze con `%pip install numpy matplotlib ipywidgets`
e rieseguire C06b. La compatibilità CPU è prevista; l'esecuzione nel servizio Colab non è attestata qui.

**Alternativa statica:** se lo slider non compare, eseguire in una nuova cella
`stereo_mostra(Z=4.0, salva=True)`; sono disponibili anche Z=1 e Z=2.
Il PNG iniziale e i salvataggi con **Salva PNG** o `stereo_salva()` vanno in
`output/stereo_profondita/`. In Colab scaricarli dal pannello File prima di chiudere il runtime.

In [ ]:
# C06b — demo autonoma; non modifica la scena o il seed del laboratorio
from pathlib import Path as _sdPath
from io import BytesIO as _sdBytesIO
import numpy as _sdnp
import matplotlib.pyplot as _sdplt
from IPython.display import display as _sddisplay, Image as _sdImage

_stereo_cartella = _sdPath.cwd() / "output" / "stereo_profondita"
_stereo_cartella.mkdir(parents=True, exist_ok=True)
_stereo_display = None
_stereo_stato = {"Z": 2.0}
_stereo_syncing = False


def stereo_valori(Z=2.0):
    """Camere parallele, origine principale, X/Y fissi; lunghezze in metri."""
    Z = float(Z)
    if not _sdnp.isfinite(Z) or Z <= 0:
        raise ValueError("Z deve essere finita e maggiore di zero, in metri.")
    f, b, X, Y = 800.0, 0.12, 0.18, 0.10
    p_L = _sdnp.array([f * X / Z, f * Y / Z])
    p_R = _sdnp.array([f * (X - b) / Z, f * Y / Z])
    d = float(p_L[0] - p_R[0])
    return {"Z": Z, "f": f, "b": b, "X": X, "Y": Y,
            "p_L": p_L, "p_R": p_R, "d": d, "Z_recuperata": f * b / d}


# Tre controlli analitici: la riga resta comune, ma cambia con Z.
for _sdZ, _sdd in ((1.0, 96.0), (2.0, 48.0), (4.0, 24.0)):
    _sdr = stereo_valori(_sdZ)
    assert _sdnp.isclose(_sdr["d"], _sdd)
    assert _sdnp.isclose(_sdr["Z_recuperata"], _sdZ)
    assert _sdnp.isclose(_sdr["p_L"][1], _sdr["p_R"][1])
assert _sdnp.allclose(stereo_valori(2)["p_L"], [72, 40])
assert _sdnp.allclose(stereo_valori(2)["p_R"], [24, 40])


def stereo_figura(Z=2.0):
    """Crea la figura senza modificare dati, generatori casuali o pose del lab."""
    s = stereo_valori(Z)
    fig = _sdplt.figure(figsize=(13.2, 9.2), layout="constrained")
    griglia = fig.add_gridspec(2, 2, height_ratios=[1.0, 1.0])
    scena = fig.add_subplot(griglia[0, 0], projection="3d")
    curva = fig.add_subplot(griglia[0, 1])
    vista_L = fig.add_subplot(griglia[1, 0])
    vista_R = fig.add_subplot(griglia[1, 1])
    colori = {"L": "#008b9a", "R": "#7651ce", "P": "#e6554a"}
    P = _sdnp.array([s["X"], s["Y"], s["Z"]])
    C_L, C_R = _sdnp.zeros(3), _sdnp.array([s["b"], 0.0, 0.0])
    for C, nome, colore in ((C_L, "L", colori["L"]), (C_R, "R", colori["R"])):
        scena.scatter(*C, s=50, color=colore)
        scena.text(C[0], C[1] - 0.018, 0, "$C_" + nome + "$", color=colore)
        scena.plot([C[0], P[0]], [C[1], P[1]], [0, P[2]], color=colore, lw=2)
        scena.plot([C[0], C[0]], [0, 0], [0, 8], color=colore, ls="--", alpha=.4)
    scena.plot([0, s["b"]], [0, 0], [0, 0], color="#172a3b", lw=2)
    scena.scatter(*P, s=65, color=colori["P"])
    scena.text(P[0], P[1], P[2] + 0.3, "$P$", color=colori["P"])
    scena.set(xlim=(-.025, .24), ylim=(.135, -.035), zlim=(0, 8.5),
              xlabel="X (m)", ylabel="Y (m)", zlabel="Z (m)",
              title="Punto 3D e raggi — assi con scale diverse")
    scena.set_box_aspect((1.6, 1, 1.7))
    scena.view_init(elev=18, azim=-70)
    scena.set_xticks([0, .12, .18]); scena.set_yticks([0, .10])
    scena.set_zticks([0, 2, 4, 6, 8])

    zs = _sdnp.linspace(.75, 8.0, 240)
    curva.plot(zs, s["f"] * s["b"] / zs, lw=2.5, color=colori["L"])
    curva.scatter([s["Z"]], [s["d"]], s=85, color=colori["P"], zorder=5)
    curva.plot([s["Z"], s["Z"]], [0, s["d"]], "--", color=colori["P"], alpha=.7)
    curva.plot([.75, s["Z"]], [s["d"], s["d"]], "--", color=colori["P"], alpha=.7)
    curva.set(xlim=(.65, 8.15), ylim=(0, 140), xlabel="Profondità Z (m)",
              ylabel="Disparità d (px)", title="$d=fb/Z$: più lontano, meno disparità")
    curva.grid(alpha=.2)
    curva.text(.97, .92, f"Z = {s['Z']:.2f} m\nd = {s['d']:.2f} px\nfb/d = {s['Z_recuperata']:.2f} m",
               transform=curva.transAxes, ha="right", va="top",
               bbox={"facecolor": "white", "alpha": .9, "edgecolor": "#c7d5df"})

    for ax, nome, p, colore in ((vista_L, "L", s["p_L"], colori["L"]),
                                 (vista_R, "R", s["p_R"], colori["R"])):
        ax.set(xlim=(-45, 205), ylim=(120, -35), aspect="equal",
               xlabel="$u_" + nome + "$ (px)", ylabel="$v_" + nome + "$ (px; positivo giù)",
               title=("Vista sinistra" if nome == "L" else "Vista destra") + " — origine principale")
        ax.axhline(0, color="#657789", lw=1)
        ax.axvline(0, color="#657789", lw=1)
        ax.axhline(p[1], color=colori["P"], lw=1.5, ls="--")
        ax.scatter([0], [0], color="#172a3b", s=25)
        ax.text(-5, -9, "$O_" + nome + "$", ha="right", color="#172a3b")
        ax.scatter([p[0]], [p[1]], color=colore, s=75, zorder=5)
        spostamento_etichetta = (-28, 9) if p[0] > 180 else (8, 9)
        ax.annotate("$p_" + nome + "$", p, xytext=spostamento_etichetta, textcoords="offset points",
                    color=colore, fontsize=13)
        ax.text(.96, .04, f"u{nome} = {p[0]:.2f} px; v{nome} = {p[1]:.2f} px",
                transform=ax.transAxes, ha="right", va="bottom", color=colore,
                bbox={"facecolor": "white", "alpha": .9, "edgecolor": "none"})
        ax.set_xticks([0, 40, 80, 120, 160, 200])
        ax.set_yticks([0, 20, 40, 60, 80, 100, 120]); ax.grid(alpha=.15)

    fig.suptitle(
        f"Stereo canonica: P = (0.18, 0.10, {s['Z']:.2f}) m; f = 800 px; b = 0.12 m\n"
        f"vL = vR = {s['p_L'][1]:.2f} px; d = uL − uR = {s['d']:.2f} px",
        fontsize=15)
    return fig


def stereo_salva(Z=None):
    """Salva un PNG del valore corrente, anche senza controlli interattivi."""
    if Z is None:
        Z = _stereo_stato["Z"]
    fig = stereo_figura(Z)
    destinazione = _stereo_cartella / f"stereo_profondita_Z{float(Z):.2f}m.png"
    fig.savefig(destinazione, dpi=150)
    _sdplt.close(fig)
    print("PNG salvato:", destinazione)
    return destinazione


def stereo_mostra(Z=2.0, salva=False):
    """Aggiorna la figura; utile anche come alternativa statica allo slider."""
    global _stereo_display, _stereo_syncing
    stato = stereo_valori(Z)
    _stereo_stato.update(stato)
    slider = globals().get("stereo_slider")
    if slider is not None and slider.min <= float(Z) <= slider.max and slider.value != float(Z):
        _stereo_syncing = True
        try:
            slider.value = float(Z)
        finally:
            _stereo_syncing = False
    fig = stereo_figura(Z)
    immagine = _sdBytesIO()
    fig.savefig(immagine, format="png", dpi=130)
    _sdplt.close(fig)
    contenuto = _sdImage(data=immagine.getvalue())
    if _stereo_display is None:
        _stereo_display = _sddisplay(contenuto, display_id=True)
    else:
        _stereo_display.update(contenuto)
    if salva:
        stereo_salva(Z)
    return stato


def stereo_widget_callback(change):
    """Callback dello slider: applica lo stesso renderer dell'alternativa statica."""
    if change.get("name") == "value" and not _stereo_syncing:
        stereo_mostra(Z=float(change["new"]))


# La figura statica compare sempre e resta consultabile nelle esportazioni HTML.
stereo_mostra(Z=2.0, salva=True)
print("Controlli Z=1/2/4 m superati: disparità 96/48/24 px; profondità recuperata corretta.")

try:
    import ipywidgets as _sdwidgets
    try:
        from google.colab import output as _sdcolab_output
    except ImportError:
        pass
    else:
        _sdcolab_output.enable_custom_widget_manager()
    stereo_slider = _sdwidgets.FloatSlider(
        value=2.0, min=.75, max=8.0, step=.05, description="Z (m)",
        continuous_update=False, readout_format=".2f", layout=_sdwidgets.Layout(width="70%"))
    stereo_slider.observe(stereo_widget_callback, names="value")
    stereo_bottone = _sdwidgets.Button(description="Salva PNG", icon="save")
    stereo_bottone.on_click(lambda _: stereo_salva())
    _sddisplay(_sdwidgets.HBox([stereo_slider, stereo_bottone]))
    print("Muovere Z aggiorna P, le due proiezioni e il marker sulla curva.")
except ImportError:
    print("Slider non disponibile: usare stereo_mostra(Z=4.0, salva=True).")
    print("Per attivarlo installare ipywidgets (%pip install ipywidgets), poi rieseguire C06b.")


In [ ]:
# C07
fig, ax = plt.subplots(figsize=(11, 6))
ax.set_xlim(0, W); ax.set_ylim(H, 0); ax.set_aspect('equal')
ax.set_title('Corrispondenze sovrapposte: vettori corti ≠ sempre match corretti')
sel = rng.choice(len(x1), min(70, len(x1)), replace=False)
for i in sel:
    color = '#178f78' if true_inlier[i] else '#df5b52'
    ax.plot([x1[i,0], x2[i,0]], [x1[i,1], x2[i,1]], color=color, alpha=.55, lw=1)
ax.scatter(x1[sel,0], x1[sel,1], s=13, c='#17324d', label='vista 1')
ax.scatter(x2[sel,0], x2[sel,1], s=13, c=np.where(true_inlier[sel], '#178f78', '#df5b52'),
           label='vista 2 / rosso=outlier')
ax.legend(loc='lower right'); ax.grid(alpha=.15); plt.show()

**C08**

### Interpretazione — consegna

Perché la sola lunghezza del vettore fra i due pixel non è un test geometrico affidabile?

> **Risposta:** _scrivere qui, citando almeno una misura e una figura._

**C09**

## 2. Matrice fondamentale, RANSAC e linee epipolari

Stimiamo $\mathbf F$ dai pixel. La soglia fornita a OpenCV è in pixel per
questa chiamata. Valutiamo precision e recall perché conosciamo gli outlier
inseriti; su dati reali useremmo residui, copertura e stabilità.

In [ ]:
# C10
threshold_px = 1.5
F_est, mask = cv2.findFundamentalMat(
    x1.astype(np.float64), x2.astype(np.float64),
    cv2.FM_RANSAC, threshold_px, 0.999, 10000
)
ransac_inlier = mask.ravel().astype(bool)

tp = np.sum(ransac_inlier & true_inlier)
fp = np.sum(ransac_inlier & ~true_inlier)
fn = np.sum(~ransac_inlier & true_inlier)
precision = tp / max(tp + fp, 1)
recall = tp / max(tp + fn, 1)
svals = np.linalg.svd(F_est, compute_uv=False)
dS = np.sqrt(sampson(F_est, x1, x2))

print('F stimata (scala arbitraria):\n', F_est / np.linalg.norm(F_est))
print('Valori singolari:', svals)
print(f'Inlier RANSAC: {ransac_inlier.sum()}/{len(x1)}')
print(f'Precision={precision:.3f} | Recall={recall:.3f}')
print('Sampson sqrt sugli inlier [px circa]:',
      np.quantile(dS[ransac_inlier], [0.5, 0.9, 0.95]))

In [ ]:
# C11
fig, ax = plt.subplots(figsize=(11, 6))
ax.set_xlim(0, W); ax.set_ylim(H, 0); ax.set_aspect('equal')
ax.set_title('Vista 2: punti e linee epipolari trasferite dalla vista 1')
good = np.flatnonzero(ransac_inlier)
show = good[np.linspace(0, len(good)-1, min(14, len(good))).astype(int)]
colors = plt.cm.turbo(np.linspace(0, 1, len(show)))
for i, color in zip(show, colors):
    line = F_est @ np.r_[x1[i], 1.0]
    seg = line_segment(line, W, H)
    if len(seg) == 2:
        ax.plot([seg[0][0], seg[1][0]], [seg[0][1], seg[1][1]], color=color, lw=1.2)
    ax.scatter(x2[i,0], x2[i,1], s=36, color=color, edgecolor='black', linewidth=.4)
ax.grid(alpha=.15); plt.show()

**C12**

### Interpretazione — consegna

Commenta rango, quantili Sampson, precision e recall. Perché il numero di inlier da solo non basta?

> **Risposta:** _scrivere qui, citando almeno una misura e una figura._

**C13**

## 3. Sweep della soglia e della contaminazione

La soglia non è universale. Ripetiamo la stima sulla stessa coppia variando
la soglia; poi generiamo dataset con percentuali crescenti di outlier. Per ciascuna percentuale si esegue una sola realizzazione con seed dichiarato: non è una stima della probabilità di successo o della variabilità fra ripetizioni.

In [ ]:
# C14
thresholds = np.array([0.4, 0.7, 1.0, 1.5, 2.0, 3.0, 5.0])
rows = []
for tau in thresholds:
    cv2.setRNGSeed(7)
    F, m = cv2.findFundamentalMat(x1, x2, cv2.FM_RANSAC, float(tau), 0.999, 10000)
    pred = m.ravel().astype(bool)
    tp = np.sum(pred & true_inlier); fp = np.sum(pred & ~true_inlier)
    fn = np.sum(~pred & true_inlier)
    rows.append((tau, pred.sum(), tp/max(tp+fp,1), tp/max(tp+fn,1),
                 np.median(np.sqrt(sampson(F, x1[pred], x2[pred])))))
rows = np.array(rows, float)

fig, ax = plt.subplots(1, 3, figsize=(14, 4), layout="constrained")
ax[0].plot(rows[:,0], rows[:,2], 'o-', label='precision')
ax[0].plot(rows[:,0], rows[:,3], 'o-', label='recall')
ax[0].set(xlabel='soglia RANSAC [px]', ylabel='score', ylim=(0,1.04)); ax[0].legend(); ax[0].grid(alpha=.25)
ax[1].plot(rows[:,0], rows[:,1], 'o-', color='#674ea7', label='inlier')
ax[2].plot(rows[:,0], rows[:,4], 's--', color='#e69138', label='mediana Sampson')
ax[1].set(xlabel='soglia RANSAC [px]', ylabel='Numero inlier'); ax[1].grid(alpha=.25); ax[1].legend()
ax[2].set(xlabel='soglia RANSAC [px]', ylabel='Mediana radice Sampson [px]'); ax[2].grid(alpha=.25); ax[2].legend()
plt.show()
print('tau | #inlier | precision | recall | mediana sqrt(Sampson)')
print(rows)

In [ ]:
# C15
def contaminated_pair(outlier_fraction, seed=17):
    rr = np.random.default_rng(seed)
    a = x1_clean + rr.normal(0, sigma_px, x1_clean.shape)
    b = x2_clean + rr.normal(0, sigma_px, x2_clean.shape)
    truth = np.ones(len(a), bool)
    ids = rr.choice(len(a), round(outlier_fraction*len(a)), replace=False)
    b[ids] = np.c_[rr.uniform(0,W,len(ids)), rr.uniform(0,H,len(ids))]
    truth[ids] = False
    return a, b, truth

fractions = np.array([0., .1, .2, .3, .4, .5, .6])
scores = []
for k, frac in enumerate(fractions):
    a, b, truth = contaminated_pair(frac, 100+k)
    cv2.setRNGSeed(100+k)
    F, m = cv2.findFundamentalMat(a, b, cv2.FM_RANSAC, 1.5, 0.999, 15000)
    pred = m.ravel().astype(bool)
    tp = np.sum(pred & truth); fp = np.sum(pred & ~truth); fn = np.sum(~pred & truth)
    scores.append((tp/max(tp+fp,1), tp/max(tp+fn,1), pred.mean()))
scores = np.array(scores)

plt.figure(figsize=(8,4))
plt.plot(fractions, scores[:,0], 'o-', label='precision')
plt.plot(fractions, scores[:,1], 'o-', label='recall')
plt.plot(fractions, scores[:,2], 'o--', label='frazione classificata inlier')
plt.xlabel('frazione reale di outlier'); plt.ylabel('score'); plt.ylim(0,1.04)
plt.grid(alpha=.25); plt.legend(); plt.show()

**C16**

### Interpretazione — consegna

Individua una soglia ragionevole e descrivi il compromesso. Che cosa cambia aumentando gli outlier?

> **Risposta:** _scrivere qui, citando almeno una misura e una figura._

**C17**

## 4. Essenziale, posa, cheirality e triangolazione

Usiamo le intrinseche note. `recoverPose` restituisce una traslazione di norma
unitaria: ne possiamo confrontare la direzione, non il modulo. Triangoliamo
gli inlier di E che superano il controllo di cheiralità; questa maschera non è l’intersezione con gli inlier di F. La direzione orientata viene valutata dopo la selezione del ramo fisico; il confronto senza segno è riportato separatamente.

In [ ]:
# C18
E_est, mask_E = cv2.findEssentialMat(
    x1, x2, K, method=cv2.RANSAC, prob=0.999, threshold=1.0, maxIters=20000
)
n_inliers_E_before_pose = int(mask_E.astype(bool).sum())
n_pose, R_est, t_dir_est, mask_pose = cv2.recoverPose(E_est, x1, x2, K, mask=mask_E.copy())
pose_inlier = mask_pose.ravel().astype(bool)

rot_err = rotation_error_deg(R_est, R_true)
t_err_oriented = direction_error_deg(t_dir_est, t_true)
t_err = min(direction_error_deg(t_dir_est, t_true), direction_error_deg(-t_dir_est, t_true))
print('Valori singolari di E:', np.linalg.svd(E_est, compute_uv=False))
print(f'Inlier con cheirality positiva: {n_pose}/{n_inliers_E_before_pose}')
print(f'Errore rotazione: {rot_err:.3f}°')
print(f'Errore direzione traslazione orientata dopo cheiralità: {t_err_oriented:.3f}°')
print(f'Errore asse di traslazione (segno migliore): {t_err:.3f}°')
print('Norma t restituita:', np.linalg.norm(t_dir_est))

In [ ]:
# C19
ids = np.flatnonzero(pose_inlier)
P1 = K @ np.c_[np.eye(3), np.zeros(3)]
P2 = K @ np.c_[R_est, t_dir_est]
Xh = cv2.triangulatePoints(P1, P2, x1[ids].T, x2[ids].T)
X_tri = (Xh[:3] / Xh[3]).T
x1_hat, z1_hat = project(K, np.eye(3), np.zeros(3), X_tri)
x2_hat, z2_hat = project(K, R_est, t_dir_est.ravel(), X_tri)
e1 = np.linalg.norm(x1_hat - x1[ids], axis=1)
e2 = np.linalg.norm(x2_hat - x2[ids], axis=1)
positive = (z1_hat > 0) & (z2_hat > 0)

print(f'Cheirality verificata: {positive.mean():.1%}')
print('Errore riproiezione vista 1, quantili 50/90/95%:', np.quantile(e1, [.5,.9,.95]))
print('Errore riproiezione vista 2, quantili 50/90/95%:', np.quantile(e2, [.5,.9,.95]))

fig = plt.figure(figsize=(10,5))
ax = fig.add_subplot(121, projection='3d')
ax.scatter(X_tri[:,0], X_tri[:,1], X_tri[:,2], s=7, c=z1_hat, cmap='viridis')
ax.set(title='Struttura triangolata (scala arbitraria)', xlabel='X', ylabel='Y', zlabel='Z')
ax2 = fig.add_subplot(122)
ax2.hist(np.r_[e1,e2], bins=25, color='#3c78d8', alpha=.85)
ax2.set(title='Residui di riproiezione', xlabel='errore [px]', ylabel='conteggio')
plt.tight_layout(); plt.show()

**C20**

### Interpretazione — consegna

Perché t ha norma uno? Che cosa aggiunge la cheirality rispetto alla riproiezione?

> **Risposta:** _scrivere qui, citando almeno una misura e una figura._

**C21**

## 5. Profondità stereo e propagazione dell'incertezza

Per stereo rettificato $Z=fb/d$ e
$\sigma_Z\simeq Z^2\sigma_d/(fb)$. Visualizziamo sia la curva sia una mappa
baseline–distanza.

In [ ]:
# C22
f_px = 800.0
sigma_d = 0.5
depths = np.linspace(1.0, 20.0, 300)
baselines = [0.06, 0.12, 0.24]

fig, ax = plt.subplots(1,2, figsize=(11,4))
for b in baselines:
    disparity = f_px*b/depths
    sigma_z = depths**2 * sigma_d/(f_px*b)
    ax[0].plot(depths, disparity, label=f'b={b:.2f} m')
    ax[1].plot(depths, sigma_z, label=f'b={b:.2f} m')
ax[0].set(xlabel='Z [m]', ylabel='disparità [px]', title='Disparità vs distanza')
ax[1].set(xlabel='Z [m]', ylabel='σZ [m]', title='Incertezza di primo ordine')
for a in ax: a.grid(alpha=.25); a.legend()
plt.tight_layout(); plt.show()

b_grid = np.linspace(.04,.30,90)
z_grid = np.linspace(1,20,100)
BZ, ZZ = np.meshgrid(b_grid, z_grid)
SZ = ZZ**2 * sigma_d/(f_px*BZ)
plt.figure(figsize=(9,4.5))
im = plt.pcolormesh(BZ, ZZ, SZ, shading='auto', cmap='magma', vmin=0, vmax=2)
plt.colorbar(im, label='σZ [m], saturata a 2 m')
plt.xlabel('baseline [m]'); plt.ylabel('distanza Z [m]')
plt.title('Dove il sistema stereo è metricamente informativo?'); plt.show()

**C23**

### Interpretazione — consegna

Descrivi l’effetto di raddoppiare distanza e baseline. In quale zona della mappa una misura è poco credibile?

> **Risposta:** _scrivere qui, citando almeno una misura e una figura._

**C24**

## 6. Matching stereo: texture, uniformità e minimi multipli

Costruiamo segnali 1D come righe rettificate. La riga destra è una versione
traslata di quella sinistra. Confrontiamo il costo SAD in una finestra
testurizzata, una uniforme e una periodica.

In [ ]:
# C25
n = 240
xx = np.arange(n)
texture = (0.45*np.sin(xx/3.7) + 0.25*np.sin(xx/11.0) +
           0.15*rng.normal(size=n))
periodic = np.sin(2*np.pi*xx/12)
uniform = np.ones(n)*0.2 + 0.01*rng.normal(size=n)
true_d = 13

def shifted_right(left, d):
    right = np.empty_like(left)
    right[:-d] = left[d:]
    right[-d:] = left[-1]
    return right + 0.015*rng.normal(size=len(left))

def sad_curve(left, right, u=140, half=6, dmax=35):
    ds = np.arange(0,dmax+1)
    costs = []
    a = left[u-half:u+half+1]
    for d in ds:
        b = right[u-d-half:u-d+half+1]
        costs.append(np.mean(np.abs(a-b)))
    return ds, np.array(costs)

fig, ax = plt.subplots(1,3,figsize=(13,3.8), sharey=True)
for a, left, title in zip(ax, [texture, uniform, periodic], ['testurizzata','uniforme','periodica']):
    right = shifted_right(left, true_d)
    ds, c = sad_curve(left, right)
    a.plot(ds,c,'o-',ms=3); a.axvline(true_d,color='#df5b52',ls='--',label='d vera')
    a.scatter(ds[np.argmin(c)], c.min(), s=70, facecolors='none', edgecolors='#178f78', label='minimo')
    a.set(title=title, xlabel='disparità candidata [px]'); a.grid(alpha=.25)
ax[0].set_ylabel('costo SAD medio'); ax[-1].legend(); plt.tight_layout(); plt.show()

**C26**

### Interpretazione — consegna

Confronta forma e unicità dei tre minimi. Quale caso richiede una maschera di confidenza?

> **Risposta:** _scrivere qui, citando almeno una misura e una figura._

**C27**

## 7. Matrice delle misure: centraggio e rango tre

Costruiamo una sequenza ortografica con tracce complete. Confrontiamo lo
spettro della matrice non centrata, di quella centrata per immagine e di una
matrice ottenuta con proiezione prospettica. Il rango tre è esatto soltanto
nel modello affine/ortografico centrato.

In [ ]:
# C28
rr = np.random.default_rng(71)
n_frames, n_tracks = 9, 70
X_aff = np.c_[rr.uniform(-1.4,1.4,n_tracks),
              rr.uniform(-.9,.9,n_tracks),
              rr.uniform(-.7,.7,n_tracks)]
X_aff -= X_aff.mean(axis=0)  # scelta della gauge: centroide 3D nell'origine

W_aff_rows, W_persp_rows = [], []
for i in range(n_frames):
    R_i = Rotation.from_euler('xyz', [1.5*i, -2.0*i, .8*i], degrees=True).as_matrix()
    t2_i = np.array([.18*i, -.07*i])
    x_aff_i = (R_i[:2] @ X_aff.T).T + t2_i
    W_aff_rows.extend([x_aff_i[:,0], x_aff_i[:,1]])

    # Stessa struttura traslata davanti a una camera prospettica normalizzata
    Xp = X_aff + np.array([0,0,6.0])
    tp = np.array([-.10*i, .015*i, .02*i])
    Xc = (R_i @ Xp.T + tp[:,None]).T
    xp = Xc[:,:2] / Xc[:,2:3]
    W_persp_rows.extend([xp[:,0], xp[:,1]])

W_aff_unc = np.asarray(W_aff_rows)
W_aff = W_aff_unc.copy()
for i in range(n_frames):
    W_aff[2*i:2*i+2] -= W_aff[2*i:2*i+2].mean(axis=1, keepdims=True)
W_persp = np.asarray(W_persp_rows)
for i in range(n_frames):
    W_persp[2*i:2*i+2] -= W_persp[2*i:2*i+2].mean(axis=1, keepdims=True)

# Rumore piccolo per rendere l'esperimento realistico
W_aff_noisy = W_aff + rr.normal(0, .0025, W_aff.shape)
W_unc_noisy = W_aff_unc + rr.normal(0, .0025, W_aff_unc.shape)
W_persp_noisy = W_persp + rr.normal(0, .0006, W_persp.shape)

s_unc = np.linalg.svd(W_unc_noisy, compute_uv=False)
s_aff = np.linalg.svd(W_aff_noisy, compute_uv=False)
s_p = np.linalg.svd(W_persp_noisy, compute_uv=False)

U3, s3, Vt3 = np.linalg.svd(W_aff_noisy, full_matrices=False)
W_rank3 = (U3[:,:3] * s3[:3]) @ Vt3[:3]
rel_error = np.linalg.norm(W_aff_noisy-W_rank3) / np.linalg.norm(W_aff_noisy)
M_hat = U3[:,:3] * np.sqrt(s3[:3])
S_hat = np.sqrt(s3[:3])[:,None] * Vt3[:3]

print('Centroide 3D scelto:', X_aff.mean(axis=0))
print('Primi 7 valori singolari - affine non centrata:', s_unc[:7])
print('Primi 7 valori singolari - affine centrata:', s_aff[:7])
print('Primi 7 valori singolari - prospettica centrata:', s_p[:7])
print(f'Errore relativo dell’approssimazione affine di rango 3: {rel_error:.4%}')
print('Forme M_hat e S_hat:', M_hat.shape, S_hat.shape)

fig, ax = plt.subplots(1,2,figsize=(11,4))
for s, label, marker in [(s_unc,'affine non centrata','o'),
                         (s_aff,'affine centrata','s'),
                         (s_p,'prospettica centrata','^')]:
    ax[0].semilogy(np.arange(1,len(s)+1), s/s[0], marker+'-', label=label)
ax[0].axvline(3.5,color='gray',ls='--',lw=1)
ax[0].set(xlabel='indice del valore singolare', ylabel='σk / σ1',
          title='Spettro della matrice delle misure')
ax[0].set_xticks([1,3,6,9,12,15,18])
ax[0].grid(alpha=.25); ax[0].legend()

residuo_rank3 = W_aff_noisy-W_rank3
limite_residuo = float(np.max(np.abs(residuo_rank3)))
im_residuo = ax[1].imshow(residuo_rank3, cmap='coolwarm', aspect='auto', vmin=-limite_residuo, vmax=limite_residuo)
fig.colorbar(im_residuo, ax=ax[1], label='Residuo [unità sintetiche]')
ax[1].set(xlabel='traccia', ylabel='coordinate impilate',
          title='Residuo dopo SVD troncata a rango 3')
plt.tight_layout(); plt.show()

**C29**

### Interpretazione — consegna

Perché il centraggio elimina il quarto valore dominante? Perché lo stesso salto non è esatto nel caso prospettico?

> **Risposta:** _scrivere qui, citando almeno una misura e una figura._

**C30**

## 8. Mini bundle adjustment multi-vista

Tre camere osservano punti comuni. Teniamo fisse tutte le rotazioni e le pose complete delle prime due camere, e ottimizziamo solo la traslazione della terza insieme ai punti. Questo impone più vincoli della sola rimozione della gauge. Le lunghezze sono nelle unità della scena sintetica, senza attribuire metri.
Il primo fit usa perdita quadratica; il secondo perdita Huber. Non è una
pipeline SfM completa, ma rende visibile la struttura camera–punto del BA.

In [ ]:
# C31
rr = np.random.default_rng(33)
npts = 35
X_ba_true = np.c_[rr.uniform(-1.3,1.3,npts), rr.uniform(-.8,.8,npts), rr.uniform(3.5,7,npts)]
Rcams = [np.eye(3), np.eye(3), Rotation.from_euler('y', 2.0, degrees=True).as_matrix()]
tcams_true = [np.zeros(3), np.array([-.25,0,0]), np.array([-.48,.02,.015])]
obs = []
for R, t in zip(Rcams, tcams_true):
    uv, _ = project(K,R,t,X_ba_true)
    obs.append(uv + rr.normal(0,.45,uv.shape))
obs = np.asarray(obs)

# Cinque osservazioni grossolanamente errate nella terza camera
bad = rr.choice(npts,5,replace=False)
obs[2,bad] += rr.normal(0,18,(len(bad),2))

X0 = X_ba_true + rr.normal(0,.12,X_ba_true.shape)
t3_0 = tcams_true[2] + np.array([.06,-.035,.045])
p0 = np.r_[t3_0, X0.ravel()]

def ba_residual(p):
    t3 = p[:3]
    X = p[3:].reshape(-1,3)
    ts = [tcams_true[0], tcams_true[1], t3]
    residuals = []
    for k in range(3):
        uv, _ = project(K,Rcams[k],ts[k],X)
        residuals.append((uv-obs[k]).ravel())
    return np.concatenate(residuals)

r0 = ba_residual(p0)
fit_l2 = least_squares(ba_residual,p0,loss='linear',max_nfev=100,verbose=0)
fit_huber = least_squares(ba_residual,p0,loss='huber',f_scale=1.5,max_nfev=120,verbose=0)

def report_fit(name, fit):
    t3 = fit.x[:3]
    X = fit.x[3:].reshape(-1,3)
    rmse = np.sqrt(np.mean(ba_residual(fit.x)**2))
    return name, rmse, np.linalg.norm(t3-tcams_true[2]), np.median(np.linalg.norm(X-X_ba_true,axis=1))

reports = [
    ('iniziale', np.sqrt(np.mean(r0**2)), np.linalg.norm(t3_0-tcams_true[2]),
     np.median(np.linalg.norm(X0-X_ba_true,axis=1))),
    report_fit('L2',fit_l2), report_fit('Huber',fit_huber)
]
print('metodo | reproj RMSE [px] | errore t3 | mediana errore punti')
for row in reports: print(f'{row[0]:8s} | {row[1]:8.3f} | {row[2]:8.4f} | {row[3]:8.4f}')

res_l2 = ba_residual(fit_l2.x).reshape(3,npts,2)
res_h = ba_residual(fit_huber.x).reshape(3,npts,2)
plt.figure(figsize=(8,4))
plt.plot(np.linalg.norm(res_l2[2],axis=1),'o',label='L2, camera 3')
plt.plot(np.linalg.norm(res_h[2],axis=1),'o',label='Huber, camera 3')
plt.axhline(3,color='gray',ls='--',lw=1)
plt.xlabel('indice punto'); plt.ylabel('errore riproiezione [px]')
plt.title('Una perdita robusta non nasconde gli outlier: ne limita l’influenza')
plt.grid(alpha=.25); plt.legend(); plt.show()

**C32**

### Interpretazione — consegna

Perché abbiamo fissato due camere? Confronta L2 e Huber senza guardare solo il costo finale.

> **Risposta:** _scrivere qui, citando almeno una misura e una figura._

**C33**

## 9. Attività opzionale con COLMAP

Questa attività richiede un'installazione separata di COLMAP e 20–40 immagini
acquisite dallo studente. Non viene eseguita automaticamente nel notebook.

```bash
colmap feature_extractor --database_path progetto/database.db --image_path progetto/images
colmap exhaustive_matcher --database_path progetto/database.db
mkdir -p progetto/sparse
colmap mapper --database_path progetto/database.db --image_path progetto/images --output_path progetto/sparse
```

Confronta poi matcher esaustivo e sequenziale usando database e progetti separati, per evitare la contaminazione dei match. Registra versione, immagini
registrate, componenti, punti 3D, lunghezza delle tracce, errore di
riproiezione e intrinseche stimate. Il file PLY da solo non è una relazione.

**C34**

## 10. Discussione conclusiva

Consegna una breve relazione contenente:

1. una figura di linee epipolari con soglia e numero di inlier;
2. una tabella dello sweep della soglia;
3. errore di rotazione, direzione della traslazione e quantili di riproiezione;
4. una frase quantitativa sull'incertezza in profondità;
5. lo spettro della matrice delle misure prima e dopo il centraggio;
6. il confronto dei costi SAD e del BA robusto;
7. un failure mode che il solo errore medio non rivelerebbe.

**Domanda finale:** quale singolo controllo aggiungeresti per primo a una
pipeline reale e perché?